# Data Center Energy Demand

In [1]:
import os
import json
import pandas as pd

In [2]:
GOOGLE_DATA_PATH = '/content/gdrive/MyDrive/Drexel Stuff/Summer 2026/DSCI591/DSCI 591 - Capstone/Data/Energy Data'
DATA_PATH = f"{os.getcwd()}/Data"
elec_path = f"{DATA_PATH}/ELEC.txt"

In [3]:
with open(elec_path, "r") as file:
        text = file.read()

decoder = json.JSONDecoder()
idx = 0
objects = []

while idx < len(text):
    # Skip whitespace
    while idx < len(text) and text[idx].isspace():
        idx += 1

    if idx >= len(text):
        break

    obj, end = decoder.raw_decode(text, idx)
    objects.append(obj)
    idx = end

print(f"Loaded {len(objects)} JSON objects.")
print(objects[0])  # First dictionary
    

Loaded 787668 JSON objects.
{'series_id': 'ELEC.CUSTOMERS.MT-ALL.A', 'name': 'Number of customer accounts : Montana : all sectors : annual', 'units': 'number of customers', 'f': 'A', 'description': 'All end-use sectors that consume electricity; All end-use sectors that consume electricity; ', 'copyright': 'None', 'source': 'EIA, U.S. Energy Information Administration', 'iso3166': 'USA-MT', 'geography': 'USA-MT', 'start': '2008', 'end': '2026', 'last_updated': '2026-06-25T02:00:58-04:00', 'geoset_id': 'ELEC.CUSTOMERS.ALL.A', 'data': [['2026', 725965.25], ['2025', 721047.9166666666], ['2024', 692536.3333333334], ['2023', 677495.4166666666], ['2022', 667301], ['2021', 655838.0833333334], ['2020', 644771.75], ['2019', 636796], ['2018', 628134.0833333334], ['2017', 620348.9166666666], ['2016', 612919.3333333334], ['2015', 605452.75], ['2014', 597765.75], ['2013', 587168.1666666666], ['2012', 581829], ['2011', 577348.8333333334], ['2010', 574223.4166666666], ['2009', 571882.3333333334], ['20

In [24]:
def get_timeseries_df(elec_type):
    annual_rows = []
    quarterly_rows = []
    monthly_rows = []

    for obj in objects:
        series_id = obj.get("series_id", "")

        if not series_id.startswith(elec_type):
            continue

        sector_map = {
        "ALL": "All",
        "RES": "Residential",
        "COM": "Commercial",
        "IND": "Industrial",
        "TRA": "Transportation",
    }

        # Extract sector abbreviation from the series_id
        sector_code = series_id.split(".")[2].split("-")[-1]

        # Convert to full name
        sector = sector_map.get(sector_code, sector_code)

        row = {
            "series_id": series_id,
            "name": obj.get("name"),
            "geography": obj.get("geography"),
            "iso3166": obj.get("iso3166"),
            "sector": sector,          # NEW COLUMN
            "units": obj.get("units"),
        }

        for period, value in obj.get("data", []):
            row[period] = value

        if series_id.endswith(".A"):
            annual_rows.append(row)
        elif series_id.endswith(".Q"):
            quarterly_rows.append(row)
        elif series_id.endswith(".M"):
            monthly_rows.append(row)
        else:
            print(f"Unknown frequency: {series_id}")

    annual_df = pd.DataFrame(annual_rows)
    quarterly_df = pd.DataFrame(quarterly_rows)
    monthly_df = pd.DataFrame(monthly_rows)


    def sort_time_columns(df):
        metadata = [
            "series_id",
            "name",
            "geography",
            "iso3166",
            "sector",
            "units",
        ]
        time_cols = sorted([c for c in df.columns if c not in metadata])
        return df[metadata + time_cols]


    annual_df = sort_time_columns(annual_df)
    quarterly_df = sort_time_columns(quarterly_df)
    monthly_df = sort_time_columns(monthly_df)

    print(f"For {elec_type:}")
    print(f"\tAnnual:    {annual_df.shape}")
    print(f"\tQuarterly: {quarterly_df.shape}")
    print(f"\tMonthly:   {monthly_df.shape}\n")

    annual_df.to_csv(f"{DATA_PATH}/{elec_type}.A.csv")
    quarterly_df.to_csv(f"{DATA_PATH}/{elec_type}.Q.csv")
    monthly_df.to_csv(f"{DATA_PATH}/{elec_type}.M.csv")

In [29]:
elec_array = ["ELEC.CUSTOMERS", "ELEC.PRICE", "ELEC.PLANT.GEN"]
for elec_type in elec_array:
    get_timeseries_df(elec_type)

For ELEC.CUSTOMERS
	Annual:    (289, 25)
	Quarterly: (289, 80)
	Monthly:   (289, 226)

For ELEC.PRICE
	Annual:    (352, 32)
	Quarterly: (352, 108)
	Monthly:   (352, 310)

For ELEC.PLANT.GEN
	Annual:    (57485, 32)
	Quarterly: (57540, 108)
	Monthly:   (57550, 310)

